# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 05: Cortex AI Functions for Unstructured Data

---

### What You'll Learn
- **AI_CLASSIFY** - Categorize support tickets by department
- **AI_EXTRACT** - Extract entities (account numbers, amounts, dates) from free text
- **AI_SENTIMENT** - Score customer satisfaction from ticket descriptions
- **AI_SUMMARIZE_AGG** - Create aggregated summaries across multiple tickets
- **AI_COMPLETE** - Custom LLM prompts for banking-specific tasks

### Why Cortex AI SQL Functions?
These functions bring LLM capabilities directly into your SQL pipeline - no external API calls, no data movement, no infrastructure to manage. Process unstructured data at scale alongside your structured analytics.

> **Warehouse:** `HOL_MEDIUM_WH` (Cortex AI functions run on standard warehouses)

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

In [ ]:
-- Quick look at our support tickets data
SELECT TICKET_ID, CATEGORY, SUBJECT, LEFT(DESCRIPTION, 100) AS DESCRIPTION_PREVIEW
FROM SUPPORT_TICKETS
LIMIT 5;

## Step 1: AI_CLASSIFY - Automatic Ticket Categorization

Classify support tickets into departments without any training data. The model understands the categories from their names alone.

In [ ]:
-- =============================================================================
-- AI_CLASSIFY: Route tickets to the correct department
-- Zero-shot classification - no training needed!
-- =============================================================================

SELECT 
    TICKET_ID,
    SUBJECT,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        DESCRIPTION,
        ['Billing & Payments', 'Technical Support', 'Fraud & Security', 
         'Account Management', 'Loan Services', 'Credit Card Services']
    ) AS AI_CLASSIFICATION
FROM SUPPORT_TICKETS
LIMIT 20;

In [ ]:
-- Classify tickets and see distribution of AI-assigned categories
WITH classified AS (
    SELECT 
        TICKET_ID,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            DESCRIPTION,
            ['Billing & Payments', 'Technical Support', 'Fraud & Security', 
             'Account Closure', 'General Inquiry', 'Dispute Resolution']
        ):labels[0]::VARCHAR AS AI_CATEGORY
    FROM SUPPORT_TICKETS
    SAMPLE (1000 ROWS)
)
SELECT 
    AI_CATEGORY,
    COUNT(*) AS TICKET_COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PCT
FROM classified
GROUP BY AI_CATEGORY
ORDER BY TICKET_COUNT DESC;

## Step 2: AI_EXTRACT - Entity Extraction from Free Text

Pull structured data (account numbers, dollar amounts, dates, names) from unstructured ticket descriptions.

In [ ]:
-- =============================================================================
-- AI_EXTRACT: Pull structured entities from unstructured text
-- =============================================================================

SELECT 
    TICKET_ID,
    LEFT(DESCRIPTION, 80) AS DESCRIPTION_PREVIEW,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        DESCRIPTION,
        ['account_number', 'dollar_amount', 'date', 'merchant_name', 'person_name']
    ) AS EXTRACTED_ENTITIES
FROM SUPPORT_TICKETS
LIMIT 15;

In [ ]:
-- Extract and flatten for analysis - focus on dollar amounts in fraud tickets
WITH extracted AS (
    SELECT 
        TICKET_ID,
        CATEGORY,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            DESCRIPTION,
            ['dollar_amount', 'account_number']
        ) AS AI_RESULT
    FROM SUPPORT_TICKETS
    WHERE CATEGORY = 'FRAUD'
    LIMIT 10
)
SELECT 
    TICKET_ID,
    CATEGORY,
    AI_RESULT:response:dollar_amount::VARCHAR AS EXTRACTED_AMOUNT,
    AI_RESULT:response:account_number::VARCHAR AS EXTRACTED_ACCOUNT
FROM extracted;

## Step 3: AI_SENTIMENT - Customer Satisfaction Scoring

Analyze the emotional tone of customer communications to identify unhappy customers.

In [ ]:
-- =============================================================================
-- AI_SENTIMENT: Score customer satisfaction by aspect
-- Returns per-category sentiment labels: positive, negative, neutral, mixed
-- =============================================================================

SELECT 
    TICKET_ID,
    CATEGORY,
    PRIORITY,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(
        DESCRIPTION,
        ['customer_service', 'resolution_speed', 'product_quality', 'communication']
    ) AS SENTIMENT_SCORES,
    LEFT(DESCRIPTION, 60) AS DESCRIPTION_PREVIEW
FROM SUPPORT_TICKETS
SAMPLE (100 ROWS)
LIMIT 20;

In [ ]:
-- Sentiment by category - which ticket types have the angriest customers?
WITH sentiment_data AS (
    SELECT 
        CATEGORY,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(
            DESCRIPTION,
            ['customer_service', 'resolution_speed', 'product_quality', 'communication']
        ) AS SENTIMENT_RAW
    FROM SUPPORT_TICKETS
    SAMPLE (500 ROWS)
)
SELECT 
    CATEGORY,
    COUNT(*) AS TICKET_COUNT,
    SENTIMENT_RAW:categories[0]:sentiment::VARCHAR AS CUSTOMER_SERVICE_SENTIMENT
FROM sentiment_data
GROUP BY CATEGORY, CUSTOMER_SERVICE_SENTIMENT
ORDER BY CATEGORY, CUSTOMER_SERVICE_SENTIMENT;

## Step 4: AI_SUMMARIZE_AGG - Aggregate Summaries Across Groups

`AI_SUMMARIZE_AGG` summarizes multiple text values within a group into a single coherent summary. Unlike AI_COMPLETE (which handles one row), this is an **aggregate function** — it reads all rows in a group and produces one summary per group.

In [ ]:
-- =============================================================================
-- AI_SUMMARIZE_AGG: Summarize all ticket descriptions per category
-- Produces one concise summary per ticket category
-- =============================================================================

SELECT 
    CATEGORY,
    COUNT(*) AS ticket_count,
    AI_SUMMARIZE_AGG(DESCRIPTION) AS CATEGORY_SUMMARY
FROM SUPPORT_TICKETS
GROUP BY CATEGORY
ORDER BY ticket_count DESC;

## Step 5: AI_COMPLETE - Custom LLM Prompts for Banking Tasks

Use custom prompts to handle banking-specific tasks: generate responses, extract action items, assess urgency.

In [ ]:
-- =============================================================================
-- AI_COMPLETE: Custom prompts for banking-specific tasks
-- =============================================================================

-- Task 1: Generate customer response drafts
SELECT 
    TICKET_ID,
    CATEGORY,
    SUBJECT,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        CONCAT(
            'You are a professional customer service agent at a major US bank. ',
            'Write a brief, empathetic response to this customer support ticket. ',
            'Be professional and offer a clear next step. Keep it under 100 words.\n\n',
            'Customer message: ', DESCRIPTION
        )
    ) AS SUGGESTED_RESPONSE
FROM SUPPORT_TICKETS
SAMPLE (5 ROWS);

In [ ]:
-- Task 3: Urgency assessment with reasoning
SELECT 
    TICKET_ID,
    PRIORITY AS ASSIGNED_PRIORITY,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        CONCAT(
            'Assess the urgency of this banking support ticket on a scale of 1-5 ',
            '(1=low, 5=critical). Respond with JSON: {"urgency": N, "reason": "brief explanation"}\n\n',
            'Ticket: ', DESCRIPTION
        )
    ) AS AI_URGENCY_ASSESSMENT
FROM SUPPORT_TICKETS
SAMPLE (10 ROWS);

## Step 6: Build an AI-Enriched Support Tickets Table

Combine all Cortex AI outputs into a single enriched table for dashboards and analytics.

In [ ]:
-- =============================================================================
-- BUILD AI-ENRICHED TABLE (run on a sample for demo speed)
-- In production, run on full table with a larger warehouse
-- =============================================================================

CREATE OR REPLACE TABLE SUPPORT_TICKETS_AI_ENRICHED AS
WITH enriched AS (
    SELECT 
        TICKET_ID,
        CUSTOMER_ID,
        CREATED_AT,
        SUBJECT,
        DESCRIPTION,
        PRIORITY,
        CHANNEL,
        STATUS,
        -- AI Enrichments (call once, parse below)
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            DESCRIPTION,
            ['Billing & Payments', 'Technical Support', 'Fraud & Security', 
             'Account Closure', 'General Inquiry', 'Dispute Resolution']
        ) AS CLASSIFY_RAW,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(DESCRIPTION) AS SENTIMENT_RAW,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            DESCRIPTION,
            ['dollar_amount', 'account_number', 'date']
        ) AS EXTRACT_RAW,
        CURRENT_TIMESTAMP() AS ENRICHED_AT
    FROM SUPPORT_TICKETS
    SAMPLE (1000 ROWS)
)
SELECT 
    TICKET_ID,
    CUSTOMER_ID,
    CREATED_AT,
    SUBJECT,
    DESCRIPTION,
    PRIORITY,
    CHANNEL,
    STATUS,
    CLASSIFY_RAW:labels[0]::VARCHAR AS AI_CATEGORY,
    SENTIMENT_RAW:categories[0]:sentiment::VARCHAR AS SENTIMENT_SCORE,
    EXTRACT_RAW:response:dollar_amount::VARCHAR AS DOLLAR_AMOUNT,
    EXTRACT_RAW:response:account_number::VARCHAR AS ACCOUNT_NUMBER,
    ENRICHED_AT
FROM enriched;

SELECT * FROM SUPPORT_TICKETS_AI_ENRICHED LIMIT 10;

In [ ]:
-- Mask PII in ticket descriptions using AI_REDACT
-- Automatically detects and replaces names, account numbers, phone numbers, etc.
SELECT 
    TICKET_ID,
    DESCRIPTION,
    SNOWFLAKE.CORTEX.AI_REDACT(
        input => DESCRIPTION,
        categories => ['NAME', 'EMAIL', 'PHONE_NUMBER', 'PAYMENT_CARD_DATA', 'ADDRESS', 'NATIONAL_ID']
    ) AS REDACTED_DESCRIPTION
FROM SUPPORT_TICKETS
SAMPLE (5 ROWS);

## Summary

| Cortex AI Function | Use Case | Input | Output |
|---|---|---|---|
| **AI_CLASSIFY** | Route tickets to departments | Text + category list | Predicted category + confidence |
| **AI_EXTRACT** | Pull entities from free text | Text + entity names | JSON with extracted values |
| **AI_SENTIMENT** | Customer satisfaction scoring | Text | Score from -1 to 1 |
| **AI_SUMMARIZE_AGG** | Executive digests | Multiple texts (grouped) | Single summary paragraph |
| **AI_COMPLETE** | Custom prompts | Text + instruction | Free-form LLM response |

**Key Benefits:**
- No training data needed (zero-shot)
- Runs in SQL alongside your structured analytics
- Scales with your warehouse
- Data never leaves Snowflake (governance preserved)

---

**Next:** Proceed to `06_Observability_Error_Tables.ipynb` for production monitoring patterns.